In [0]:
!pip install openpyxl 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import os
from datetime import datetime

input_path = '/Volumes/techsolve/ticket_bronze/raw_file/Input/'
output_base_path = '/Volumes/techsolve/ticket_bronze/raw_file/output/'

folder_name = datetime.now().strftime('%Y-%m-%d_%H%M%S')
output_path = os.path.join(output_base_path, folder_name)
os.makedirs(output_path, exist_ok=True)

for file in os.listdir(input_path):
    if file.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(os.path.join(input_path, file))
        csv_name = file.replace('.xlsx', '.csv').replace('.xls', '.csv')
        df.to_csv(os.path.join(output_path, csv_name), index=False)

print(f"Files written to: {output_path}")

Files written to: /Volumes/data_dev/nithin_dev/test/output/2026-07-17_230907


In [0]:
%python
df = (spark.readStream
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv") \
 .option("header", "true")
 .option("cloudFiles.inferColumnTypes", "true")
 .option("cloudFiles.SchemaLocation","/Volumes/techsolve/ticket_bronze/raw_file/output/_schema")
 .option("cloudFiles.schemaEvolutionMode","none")
 .load("/Volumes/techsolve/ticket_bronze/raw_file/output/")
 )

query = (df.writeStream.format("delta")
    .option("checkpointLocation", "/Volumes/techsolve/ticket_bronze/raw_file/output/_checkpoint")
    .trigger(availableNow=True)
    .table("techsolve.ticket_bronze.ticket_bronze")
 )
query.awaitTermination()